# HODGKIN-HUXLEY NEURON MODEL

Concepts:
- Ion channel dynamics
- Action potential generation
- Biophysical neuron modeling

Visualization:
Plotly interactive figures

## Import libraries

In [2]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

## Hodgkin-Huxley Neuron Class

In [3]:
class HodgkinHuxleyNeuron:

    def __init__(self, dt=0.01, T=50):

        self.dt = dt
        self.T = T
        self.time = np.arange(0, T, dt)

        # Membrane properties
        self.Cm = 1.0

        # Conductances
        self.gNa = 120
        self.gK = 36
        self.gL = 0.3

        # Reversal potentials
        self.ENa = 50
        self.EK = -77
        self.EL = -54.4

        # Initialize variables
        self.V = np.zeros(len(self.time))
        self.m = np.zeros(len(self.time))
        self.h = np.zeros(len(self.time))
        self.n = np.zeros(len(self.time))

        self.V[0] = -65
        self.m[0] = 0.05
        self.h[0] = 0.6
        self.n[0] = 0.32


    # Channel Rate Functions
    def alpha_m(self, V):
        return 0.1*(V+40)/(1-np.exp(-(V+40)/10))

    def beta_m(self, V):
        return 4*np.exp(-(V+65)/18)

    def alpha_h(self, V):
        return 0.07*np.exp(-(V+65)/20)

    def beta_h(self, V):
        return 1/(1+np.exp(-(V+35)/10))

    def alpha_n(self, V):
        return 0.01*(V+55)/(1-np.exp(-(V+55)/10))

    def beta_n(self, V):
        return 0.125*np.exp(-(V+65)/80)



    # Input Current
    def input_current(self, t):

        if 10 <= t <= 40:
            return 10
        return 0

    # Simulation
    def simulate(self):

        INa = np.zeros(len(self.time))
        IK = np.zeros(len(self.time))
        IL = np.zeros(len(self.time))
        I_input = np.zeros(len(self.time))

        for i in range(1, len(self.time)):

            V = self.V[i-1]

            I = self.input_current(self.time[i])
            I_input[i] = I

            # Update gating variables

            m = self.m[i-1]
            h = self.h[i-1]
            n = self.n[i-1]

            dm = self.alpha_m(V)*(1-m) - self.beta_m(V)*m
            dh = self.alpha_h(V)*(1-h) - self.beta_h(V)*h
            dn = self.alpha_n(V)*(1-n) - self.beta_n(V)*n

            m = m + self.dt*dm
            h = h + self.dt*dh
            n = n + self.dt*dn

            self.m[i] = m
            self.h[i] = h
            self.n[i] = n

            # Channel currents

            INa[i] = self.gNa*(m**3)*h*(V-self.ENa)
            IK[i] = self.gK*(n**4)*(V-self.EK)
            IL[i] = self.gL*(V-self.EL)

            # Voltage update

            dVdt = (I - INa[i] - IK[i] - IL[i]) / self.Cm
            self.V[i] = V + self.dt*dVdt

        return self.time, self.V, self.m, self.h, self.n, INa, IK, IL, I_input


## Visualization

In [4]:
def plot_results(time, V, m, h, n, INa, IK, IL, I_input):

    fig = make_subplots(
        rows=4,
        cols=1,
        subplot_titles=(
            "Membrane Potential",
            "Gating Variables",
            "Ion Channel Currents",
            "Input Current"
        )
    )

    fig.add_trace(go.Scatter(x=time,y=V,name="Voltage"),row=1,col=1)

    fig.add_trace(go.Scatter(x=time,y=m,name="m"),row=2,col=1)
    fig.add_trace(go.Scatter(x=time,y=h,name="h"),row=2,col=1)
    fig.add_trace(go.Scatter(x=time,y=n,name="n"),row=2,col=1)

    fig.add_trace(go.Scatter(x=time,y=INa,name="Na Current"),row=3,col=1)
    fig.add_trace(go.Scatter(x=time,y=IK,name="K Current"),row=3,col=1)
    fig.add_trace(go.Scatter(x=time,y=IL,name="Leak Current"),row=3,col=1)

    fig.add_trace(go.Scatter(x=time,y=I_input,name="Input Current"),row=4,col=1)

    fig.update_layout(
        height=900,
        title="Hodgkin-Huxley Neuron Simulation",
        template="plotly_white"
    )

    fig.show()



## Main function

In [5]:
def main():

    neuron = HodgkinHuxleyNeuron()

    results = neuron.simulate()

    plot_results(*results)


if __name__ == "__main__":
    main()